In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import mteb
from mteb.benchmarks import get_benchmark

In [34]:
VN_MTEB_BENCHMARK = 'VN-MTEB (vie, v1)'
vn_benchmark = mteb.benchmarks.get_benchmark(VN_MTEB_BENCHMARK)
all_tasks = list(vn_benchmark.tasks)

task_groups = {
    "STS":                [t for t in all_tasks if t.metadata.type == "STS"],
    "Retrieval":          [t for t in all_tasks if t.metadata.type == "Retrieval"],
    "Reranking":          [t for t in all_tasks if t.metadata.type == "Reranking"],
    "Classification":     [t for t in all_tasks if t.metadata.type == "Classification"],
    "Clustering":         [t for t in all_tasks if t.metadata.type == "Clustering"],
    "PairClassification": [t for t in all_tasks if t.metadata.type == "PairClassification"],
}

for name, tasks in task_groups.items():
    print(f"{name}: {len(tasks)} tasks")

STS: 3 tasks
Retrieval: 24 tasks
Reranking: 3 tasks
Classification: 12 tasks
Clustering: 5 tasks
PairClassification: 3 tasks


In [ ]:
import os
from sentence_transformers import SentenceTransformer

MODEL_NAMES = [
    'google/embeddinggemma-300m',
    os.path.abspath('../model/gemma300-vi-trimmed-fp16'),
]

# Change to any key: "STS", "Retrieval", "Reranking", "Classification", "Clustering", "PairClassification"
RUN_TASKS = task_groups["STS"]

all_results = {}
for model_name in MODEL_NAMES:
    print(f"\n=== Evaluating: {model_name} ===")
    model = SentenceTransformer(model_name)
    all_results[model_name] = mteb.evaluate(
        model=model,
        tasks=RUN_TASKS,
        encode_kwargs={"batch_size": 16, 'show_progress_bar': True},
        show_progress_bar=True,
        num_proc=4,
        cache=mteb.cache.ResultCache("../vn_mteb_results"),
    )

SyntaxError: invalid syntax. Perhaps you forgot a comma? (427548546.py, line 17)

In [ ]:
import pandas as pd

labels = list(all_results.keys())
rows = []
for model_name, results in all_results.items():
    df = results.to_dataframe()
    score_col = df.columns[-1]
    for _, row in df.iterrows():
        rows.append({'model': model_name, 'task': row['task_name'], 'score': row[score_col]})

df = pd.DataFrame(rows).pivot(index='task', columns='model', values='score')

# rename columns to short labels
col_map = {labels[0]: 'original', labels[-1]: 'trimmed'}
df = df.rename(columns=col_map)

if 'trimmed' in df.columns and 'original' in df.columns:
    df['diff'] = df['trimmed'] - df['original']

print(df.to_string())

model             trimmed  original      diff
task                                         
BIOSSES-VN       0.778217  0.783512 -0.005295
SICK-R-VN        0.768511  0.788126 -0.019615
STSBenchmark-VN  0.783126  0.819424 -0.036298


In [30]:
from transformers import AutoTokenizer, AutoModel

for model_name in MODEL_NAMES:
    label = 'original' if model_name == MODEL_NAMES[0] else 'trimmed'
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    total = sum(p.numel() for p in model.parameters())
    embed = model.get_input_embeddings().weight.numel()
    print(f"[{label}] vocab: {tok.vocab_size:,} | total: {total/1e6:.1f}M | embed: {embed/1e6:.1f}M | other: {(total-embed)/1e6:.1f}M")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}


[original] vocab: 262,144 | total: 302.9M | embed: 201.3M | other: 101.5M


Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

[trimmed] vocab: 47,465 | total: 87.2M | embed: 36.5M | other: 50.8M
